# Regressão múltipla, categorias e controle estatístico

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_07/02_regressao_multipla_e_categorias.ipynb)

Um modelo múltiplo estima uma associação enquanto mantém matematicamente fixas as
demais variáveis incluídas. Isso responde a uma pergunta mais específica que a
regressão simples, mas não corrige automaticamente seleção, mensuração ruim ou
variáveis omitidas.

![Uma resposta recebe setas de fatores observados; uma categoria de referência ancora as comparações regionais.](imagens/02_controle_referencia.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_07'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 1. De uma associação bruta a associações condicionadas

Incluiremos urbanização, escolaridade e distância. Para cada coeficiente, a leitura
será: diferença média prevista na resposta associada a uma unidade da variável,
**entre casos iguais nos outros termos do modelo**. Essa igualdade é matemática;
sua plausibilidade depende de haver casos comparáveis no corpus.

$$Y_i = \beta_0 + \beta_1 renda_i + \beta_2 urbanizacao_i
+ \beta_3 escolaridade_i + \beta_4 distancia_i + \varepsilon_i$$

Reutilizaremos a mesma função do notebook anterior. Ela executa a álgebra para que
a atenção permaneça na especificação e na interpretação; não é um exercício de
programação matricial.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.read_csv("dados/municipios_culturais.csv")

In [ ]:
def ajustar_ols(X, y, nomes):
    X_com_intercepto = np.column_stack([np.ones(len(X)), np.asarray(X, dtype=float)])
    coeficientes = np.linalg.lstsq(X_com_intercepto, np.asarray(y, dtype=float), rcond=None)[0]
    previstos = X_com_intercepto @ coeficientes
    residuos = np.asarray(y, dtype=float) - previstos
    r2 = 1 - np.sum(residuos ** 2) / np.sum((y - np.mean(y)) ** 2)
    return {
        "X": X_com_intercepto, "coeficientes": coeficientes,
        "previstos": previstos, "residuos": residuos,
        "r2": r2, "rmse": np.sqrt(np.mean(residuos ** 2)),
        "mae": np.mean(np.abs(residuos)), "nomes": ["intercepto", *nomes],
    }

def tabela_coeficientes(modelo):
    return pd.DataFrame({"termo": modelo["nomes"], "coeficiente": modelo["coeficientes"]})

In [ ]:
numericas = ["renda_pc", "urbanizacao_pct", "escolaridade_media_anos", "distancia_capital_km"]
multiplo = ajustar_ols(dados[numericas], dados["equipamentos_culturais_100mil"], numericas)
comparacao = pd.DataFrame({
    "modelo": ["simples", "múltiplo"],
    "R²": [ajustar_ols(dados[["renda_pc"]], dados["equipamentos_culturais_100mil"], ["renda_pc"])["r2"], multiplo["r2"]],
    "RMSE": [ajustar_ols(dados[["renda_pc"]], dados["equipamentos_culturais_100mil"], ["renda_pc"])["rmse"], multiplo["rmse"]],
})
display(tabela_coeficientes(multiplo).round(4))
display(comparacao.round(3))

## 2. Variáveis categóricas e categoria de referência

Uma variável com cinco regiões não entra como os números 1 a 5: isso inventaria
uma ordem. Criamos indicadores 0/1. Uma região fica de fora como **referência**;
seus casos não somem, apenas formam o ponto de comparação. Aqui, por ordem
alfabética, `Centro-Oeste` será a referência.

O coeficiente de `regiao_Sudeste`, por exemplo, é a diferença prevista entre
Sudeste e Centro-Oeste para os mesmos valores das variáveis numéricas incluídas.
Trocar a referência muda os contrastes exibidos, mas não as previsões do modelo.

In [ ]:
indicadores_regiao = pd.get_dummies(dados["regiao"], prefix="regiao", drop_first=True, dtype=float)
referencia = sorted(dados["regiao"].unique())[0]
X_categorico = pd.concat([dados[numericas], indicadores_regiao], axis=1)
com_regiao = ajustar_ols(X_categorico, dados["equipamentos_culturais_100mil"], list(X_categorico.columns))
print("Categoria de referência:", referencia)
display(tabela_coeficientes(com_regiao).round(4))

## 3. Previsões substantivas, não só coeficientes

Coeficientes em escalas diferentes são difíceis de comparar. Por isso, calculamos
previsões para dois perfis transparentes. Nos dois perfis, todos os indicadores
regionais valem zero; portanto, a região é **Centro-Oeste**, a categoria de
referência. Elas são cenários do modelo, não destinos inevitáveis de municípios
reais.

In [ ]:
perfil_base = {coluna: 0.0 for coluna in X_categorico.columns}
perfil_base.update({"renda_pc": 1500, "urbanizacao_pct": 60, "escolaridade_media_anos": 9, "distancia_capital_km": 250})
perfil_alto = perfil_base.copy()
perfil_alto.update({"renda_pc": 2200, "urbanizacao_pct": 85, "escolaridade_media_anos": 11})
cenarios = pd.DataFrame([perfil_base, perfil_alto], index=["perfil A", "perfil B"])
cenarios["previsao"] = np.column_stack([np.ones(2), cenarios[X_categorico.columns]]) @ com_regiao["coeficientes"]
cenarios_exibicao = cenarios[["renda_pc", "urbanizacao_pct", "escolaridade_media_anos", "distancia_capital_km", "previsao"]].copy()
cenarios_exibicao.insert(0, "regiao", referencia)
display(cenarios_exibicao.round(2))

### U07-A03 — Atividade — controle e referência

1. Compare o coeficiente de renda nos modelos simples e múltiplo.
2. Interprete um coeficiente numérico e um regional sem dizer “efeito causal”.
3. Explique a um leitor por que Centro-Oeste não desapareceu.
4. Crie dois perfis plausíveis e compare previsões, registrando cada valor escolhido.

**Dinâmica (30 minutos):** grupos de três; uma pessoa executa, outra confere as
escalas e a terceira atua como revisora da linguagem. Troquem papéis na metade.

**Resposta:** escreva aqui.

## Passagem

Até aqui, supusemos uma mesma inclinação e um ajuste aceitável para todos os casos.
O próximo notebook testa essas duas simplificações com interação e resíduos.

**Leitura:** Gelman, Hill e Vehtari (2020) e Fox (2016), listados em
`referencias.md`.